## Function/Tool Calling for Airline Booking

In [295]:
import ollama
# import json
import pprint


In [296]:
# Function for getting the prices of a Flight
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}
MODEL = "mistral:7b"
def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown Ticket Price")
    return price
    # return f"The price of a ticket to {destination_city} is {price}"

In [ ]:
# def set_ticket_price(destination_city, price):
#     print(f"Tool called for setting ticket Price for {destination_city} as {price}")
#     if(ticket_prices.get(destination_city.lower(),"Unknown Ticket Price")):
#         pass
#     else:
#         ticket_prices[destination_city] = "$"+price
#         content = "The ticket price has been set as desired...YOOOhOOOOOO\n\n"
#         print(content)
#         return content
def set_ticket_price(destination_city, price):
    print(f"Tool called for setting ticket price for {destination_city} as {price}")
    ticket_prices[destination_city.lower()] = "$" + str(price)
    content = f"The ticket price for {destination_city} has been set to ${price}."
    print(content)
    return content


In [298]:
# Below is th dictionary structure needed to describe our function in order for the llm to match the user query to our function
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer want to travel to"
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

set_price_function = {
    "name": "set_ticket_price",
    "description": "Set the price of a ticket to the destination city when the name of destination city and the price is given as input",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer want to set the price for"
            },
            "price": {
                "type": "int",
                "description": "Price to be set for the destination city"
            }
        },
        "required": ["destination_city","price"],
        "additionalProperties": False
    }
}

In [299]:
# During the llm call we need to pass the messages with the concatenated history and also another
# essential field called the tools
# this is a essential feature that modern frontier models support even the open source one.

#You can see below we pass the function description and not the actual function
# tools = [get_ticket_price]

# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function},{"type":"function","function":set_price_function}]

In [300]:
system_prompt = """
You are a concise and courteous assistant for FlightAI. You support only two functions:

1. Get the price of a ticket to a destination city. This requires only the city name.
2. Set the ticket price for a destination city. This requires the city name and the price.

You must:
- Use the tools listed in the tools=tools field.
- Call a tool immediately once all required parameters are available.
- Do not ask for travel dates, cabin class, or any other information not defined in the tool schema.
- If the user asks anything outside these two functions, respond with: "I will not be able to help in that regard."
"""


In [301]:
# def chat(message, history):
#     history = [{"role": h["role"], "content":h["content"]} for h in history]
#     messages = [{"role":"system","content":system_prompt}] + history + [{"role":"user","content":message}]
#     response = ollama.chat(model=MODEL, messages=messages, tools=tools)
#     pprint.pprint(response.model_dump_json())

#     responses = []
#     # Now checking the finish_reason parameter of the llm's output and not the message
#     # if the finish_reason parameter is set to "tool_calls" specifically then we check which function it is calling to 
#     # in our case it is get_ticket_price using the price_function description.
#     if response.message.tool_calls:
#         for call in response.message.tool_calls:
#             print(call)
#             message = response.message.content
#             messages.append({
#                 "role":"assistant",
#                 "content":message
#                 })

#             print(call.function.name)
#             print(call.function.arguments)
#             if call.function.name == "get_ticket_price":
#                 arguments = call.function.arguments
#                 city = arguments.get('destination_city')
#                 price_details = get_ticket_price(city)
#                 responses.append({
#                     "role":"tool",
#                     "content": price_details,
#                     # "tool_call_id": call.id
#                 })
#             messages.extend(responses)
#             response = ollama.chat(model=MODEL, messages=messages)
#     return response.message.content            

In [302]:
def handle_tool_calls(message):
    responses = []
    pprint.pprint(message)
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = tool_call.function.arguments
            city = arguments.get("destination_city")
            price_details = get_ticket_price(city)
            responses.append({
                "role":"tool",
                "content": price_details
            })
        if tool_call.function.name == "set_ticket_price":                        
            arguments = tool_call.function.arguments
            city1 = arguments.get('destination_city')
            price = arguments.get('price')
            content = set_ticket_price(city1,price)
            # print(content)
            responses.append({
                "role":"tool",
                "content":content,
                # "tool_call_id":tool_call.id
            })
    return responses

In [303]:
def chat(message,history):
    history = [{"role":h["role"],"content":h["content"]} for h in history]
    messages = [{"role":"system", "content":system_prompt}] + history + [{"role":"user","content":message}]
    response = ollama.chat(model=MODEL,messages=messages,tools=tools)

    while response.message.tool_calls:
        message = response.message
        responses = handle_tool_calls(message)
        messages.append({
            "role": "assistant",
            "content": message.content
        })
        messages.extend(responses)
        response = ollama.chat(model=MODEL, messages = messages)
    return response.message.content

In [304]:
import gradio as gr

gr.ChatInterface(fn=chat,type="messages").launch()

* Running on local URL:  http://127.0.0.1:7897
* To create a public link, set `share=True` in `launch()`.


In [305]:
from openai import OpenAI
import json

In [306]:
MODEL = "mistral:7b"
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [307]:
def handle_tool_calls1(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
        if tool_call.function.name == "set_ticket_price":                        
            arguments = json.loads(tool_call.function.arguments)
            city1 = arguments.get('destination_city')
            price = arguments.get('price')
            content = set_ticket_price(city1,price)
            print(content)
            responses.append({
                "role":"tool",
                "content":content,
                "tool_call_id":tool_call.id
            })

    return responses

In [308]:
# chat function for connecting ollama using openAI client
def chat(message, history):
    history = [{"role":h["role"],"content":h["content"]} for h in history]
    messages = [{"role":"system","content":system_prompt}] + history + [{"role":"user","content":message}]
    response = openai.chat.completions.create(model=MODEL,messages = messages, tools = tools)
    pprint.pprint(response)
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls1(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL,messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [309]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7898
* To create a public link, set `share=True` in `launch()`.


In [310]:
print(ticket_prices)

{'london': '$799', 'paris': '$899', 'tokyo': '$1400', 'berlin': '$499'}


In [ ]:
print(ticket_prices)

{'london': '$799', 'paris': '$899', 'tokyo': '$1400', 'berlin': '$499'}


Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='set_ticket_price', arguments={'destination_city': 'india', 'price': 1299}))])
Tool called for setting ticket price for india as 1299


Traceback (most recent call last):
  File "/home/skrrr/Projects/llm_engineering/llms/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/skrrr/Projects/llm_engineering/llms/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/skrrr/Projects/llm_engineering/llms/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/skrrr/Projects/llm_engineering/llms/lib/python3.12/site-packages/gradio/blocks.py", line 1621, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/skrrr/Projects/llm_engineering/llms/lib/python3.12/site-packages/gradio/ut